In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from tqdm import tqdm

In [2]:
dataset = pd.read_csv("./data/imdb_dataset.csv")
test_dataset = dataset.iloc[10000:12000]
dataset = dataset.head(10000) # Shrink dataset size

In [3]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
)
X = vectorizer.fit_transform(dataset["review"])
y = dataset['sentiment'].map({"positive": 1, "negative": 0})

vocab = vectorizer.get_feature_names_out()
print(f"Number of unique words: {len(vocab)}")
print(vocab)

Number of unique words: 5000
['000' '10' '100' ... 'zombie' 'zombies' 'zone']


In [4]:
test_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
)
X_test = test_vectorizer.fit_transform(test_dataset["review"])
y_test = test_dataset['sentiment'].map({"positive": 1, "negative": 0})

test_vocab = test_vectorizer.get_feature_names_out()
print(f"Number of unique words: {len(test_vocab)}")
print(test_vocab)

Number of unique words: 5000
['00' '000' '10' ... 'zero' 'zombie' 'zombies']


## Logistic Regression

In [5]:
### Model Parameter Searching
lr_ = LogisticRegression()

parameters = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # 'liblinear' supports both l1 and l2 penalties
}

lr_grid = GridSearchCV(lr_, parameters, cv=5, scoring='accuracy')
lr_grid.fit(X, y)

print("Best Model Score:", lr_grid.best_score_)
print("Best Model Parameters:", lr_grid.best_params_)

Best Model Score: 0.8758999999999999
Best Model Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}


In [6]:
### Test Model Performance
lr = LogisticRegression(**lr_grid.best_params_)
lr.fit(X, y)
y_pred = lr.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.5295


## Decision Tree

In [7]:
### Model Parameter Searching
dt_ = DecisionTreeClassifier()

parameters = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

dt_grid = RandomizedSearchCV(dt_, parameters, cv=5, scoring='accuracy', n_iter=50, n_jobs=-1)
dt_grid.fit(X, y)

print("Best Model Score:", dt_grid.best_score_)
print("Best Model Parameters:", dt_grid.best_params_)

Best Model Score: 0.7217
Best Model Parameters: {'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 20, 'criterion': 'gini'}


In [8]:
### Test Model Performance
dt = DecisionTreeClassifier(**dt_grid.best_params_)
dt.fit(X, y)
y_pred = dt.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.4715


## Random Forest

In [9]:
### Model Parameter Searching
rf_ = RandomForestClassifier(random_state=42)

parameters = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'] 
}

rf_grid =  RandomizedSearchCV(rf_, parameters, cv=5, scoring='accuracy', n_iter=50, n_jobs=-1)
rf_grid.fit(X, y)

print("Best Model Score:", rf_grid.best_score_)
print("Best Model Parameters:", rf_grid.best_params_)

Best Model Score: 0.8440999999999999
Best Model Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None}


In [10]:
### Test Model Performance
rf = RandomForestClassifier(random_state=42, **rf_grid.best_params_)
rf.fit(X, y)
y_pred = rf.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.491


## Gradient Boost

In [11]:
### Model Parameter Searching
xgb_ = XGBClassifier(random_state=42)

parameters = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_grid = RandomizedSearchCV(xgb_, parameters, cv=5, scoring='accuracy', n_iter=20, n_jobs=-1)
xgb_grid.fit(X, y)

print("Best Model Score:", xgb_grid.best_score_)
print("Best Model Parameters:", xgb_grid.best_params_)

Best Model Score: 0.842
Best Model Parameters: {'subsample': 0.6, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.3, 'colsample_bytree': 1.0}


In [12]:
### Test Model Performance
xgb = XGBClassifier(random_state=42, **xgb_grid.best_params_)
xgb.fit(X, y)
y_pred = xgb.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.528


Okay all my model are overfitted because input vector very big but anyway best model is Logistic Regression (simple model harder to overfit)